# HeartLens AI — Colab training & experiments

Runtime: **T4 GPU** (Edit → Notebook settings). Everything below runs in 4 steps; nothing needs to be uploaded.
The dataset auto-downloads from PhysioNet; code clones from GitHub.

In [ ]:
# 1. Get the code
!git clone --depth 1 https://github.com/touhidsiddiqueeraj-bit/heartlens.git
%cd heartlens

In [ ]:
# 2. Dependencies (only the few not preinstalled in Colab)
!pip install -q wfdb fpdf2

In [ ]:
# 3. Train denoiser + 3-class classifier (Normal / APB / PVC), int8 quantize, PDF report
!python3 auto_train.py --epochs 30 --max-per-class 3000

In [ ]:
# 4. Paper experiments (Exp 1-4). Run each once; results land in results/.
%cd heart-lens-training

# Exp 1: grouped patient-level CV (overnight job — ~15 retrains)
!python3 group_kfold_eval.py --folds 5 --seeds 0,1,2 --epochs 30

# Exp 2: noise robustness Raw/Filter/AE x SNR 0-40 dB
!python3 evaluate_noise_robustness.py --epochs 30

# Exp 3: external generalization mitdb -> SVDB + afdb
!python3 external_validation.py --epochs 30

# Exp 4: FP32 vs INT8 delta per architecture + size (comparison study)
!python3 compare_models.py --epochs 30

# Calibration: temperature scaling -> writes CALIB_TEMPERATURE to firmware Config.h
!python3 calibrate.py --epochs 30 --write-config

In [ ]:
# 5. Bundle results for download (~1 MB: models, report PDF, figures, JSON)
!mkdir -p /content/out && cp -r models /content/out/ && cp -r results /content/out/
!cp ../auto_train_output/training_report.pdf /content/out/ 2>/dev/null; true
!cd /content/out && zip -qr heartlens_results.zip . && ls -lh heartlens_results.zip

## After downloading `heartlens_results.zip`
1. Unzip into the local repo: `models/` → `heart-lens-training/models/`, `results/` → `heart-lens-training/results/`.
2. Generate firmware headers locally: `cd HeartLens_Firmware && ./convert_tflite_to_headers.sh ../heart-lens-training/models models/`.
3. Hardware experiments (Exp 5-6) with the ESP32-S3: see `docs/EXPERIMENTS.md`.